### Cat vs Dog Classification Using fine-tuned VGG-16

In [1]:
!pip install numpy pandas matplotlib

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.6 MB 8.4 MB/s eta 0:00:02
   ---------- ----------------------------- 3.4/12.6 MB 8.4 MB/s eta 0:00:02
   ------------------ --------------------- 5.8/12.6 MB 9.3 MB/s eta 0:00:01
   ------------------------- -------------- 8.1/12.6 MB 9.5 MB/s eta 0:00:01
   ------------------------------- -------- 10.0/12.6 MB 9.5 MB/s eta 0:00:01
   -------------------------------------- - 12.1/12.6 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 9.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   ------- -------------------------------- 2.1/11.5 MB 10.7 MB/s eta 0:00:01
   -------------- ------------------------- 4.2/11.5 MB 10.5 MB/s eta 0:00:01
   ---------------------- ----------------- 6.6/11.5 MB 10.9 MB/s eta 0:00:01
   ------------------------------ --------- 8.7/11.5 MB 10.5 MB/s eta 0:00:01
   ---

In [21]:
!pip install tensorflow
!pip install scikit-learn

  You can safely remove it manually.
  You can safely remove it manually.



   ---------------------------------------- 0.0/390.3 MB ? eta -:--:--
   ---------------------------------------- 1.0/390.3 MB 6.3 MB/s eta 0:01:02
   ---------------------------------------- 2.4/390.3 MB 6.4 MB/s eta 0:01:01
   ---------------------------------------- 3.4/390.3 MB 5.6 MB/s eta 0:01:10
    --------------------------------------- 5.2/390.3 MB 6.2 MB/s eta 0:01:02
    --------------------------------------- 7.6/390.3 MB 7.1 MB/s eta 0:00:54
   - -------------------------------------- 10.0/390.3 MB 7.9 MB/s eta 0:00:49
   - -------------------------------------- 11.8/390.3 MB 7.9 MB/s eta 0:00:48
   - -------------------------------------- 13.6/390.3 MB 8.1 MB/s eta 0:00:47
   - -------------------------------------- 15.7/390.3 MB 8.2 MB/s eta 0:00:46
   - -------------------------------------- 17.8/390.3 MB 8.3 MB/s eta 0:00:45
   -- ------------------------------------- 20.2/390.3 MB 8.6 MB/s eta 0:00:44
   -- ------------------------------------- 21.8/390.3 MB 8.4 MB

In [9]:
!pip install opencv-python

   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   -- ------------------------------------- 2.1/38.8 MB 10.7 MB/s eta 0:00:04
   ---- ----------------------------------- 3.9/38.8 MB 9.4 MB/s eta 0:00:04
   ------ --------------------------------- 6.0/38.8 MB 9.7 MB/s eta 0:00:04
   -------- ------------------------------- 7.9/38.8 MB 9.4 MB/s eta 0:00:04
   --------- ------------------------------ 8.9/38.8 MB 8.9 MB/s eta 0:00:04
   --------- ------------------------------ 9.7/38.8 MB 7.9 MB/s eta 0:00:04
   ---------- ----------------------------- 10.5/38.8 MB 7.3 MB/s eta 0:00:04
   ------------ --------------------------- 11.8/38.8 MB 7.0 MB/s eta 0:00:04
   -------------- ------------------------- 13.9/38.8 MB 7.2 MB/s eta 0:00:04
   --------------- ------------------------ 15.2/38.8 MB 7.2 MB/s eta 0:00:04
   ----------------- ---------------------- 16.8/38.8 MB 7.1 MB/s eta 0:00:04
   ------------------ --------------------- 18.1/38.8 MB 7.1 MB/s eta 0:00:03

In [1]:
import os
import zipfile
import pandas as pd
import numpy as np
import cv2

In [2]:
image_dir = "datasets/train"


filenames = os.listdir(image_dir)


labels = [x.split(".")[0] for x in filenames]

data = pd.DataFrame({"filename": filenames, "label": labels})
data.tail()

,filename,label
19995,dog.9993.jpg,dog
19996,dog.9994.jpg,dog
19997,dog.9995.jpg,dog
19998,dog.9998.jpg,dog
19999,dog.9999.jpg,dog


In [3]:
image_dir_val = "datasets/val"
filenames_val = os.listdir(image_dir_val)

labels_val = [x.split(".")[0] for x in filenames_val]
data_val = pd.DataFrame({"filename": filenames_val, "label": labels_val})

In [4]:
# Scikit-learn
from sklearn.metrics import classification_report,confusion_matrix,ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

# Tensorflow
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau,EarlyStopping
from keras.applications import VGG16
from keras.applications.vgg16 import preprocess_input
from keras.models import Sequential,Model
from keras.layers import Flatten,Dense,Dropout

In [5]:
image_paths = [os.path.join(image_dir, filename1) for filename1 in os.listdir(image_dir)]

image_width = []
image_height = []
for image_path in image_paths:
    image = cv2.imread(image_path)
    height, width, _ = image.shape
    image_width.append(width)
    image_height.append(height)

median_width = np.median(image_width)
median_height = np.median(image_height)


print('median_size:', median_width,  'X', median_height)

median_size: 448.0 X 374.0


In [6]:
batch_size = 64
size = (370, 370)

# Create image data generator
idg = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function = tf.keras.applications.vgg16.preprocess_input)

train_idg = idg.flow_from_dataframe(data, "datasets/train/", x_col= "filename", y_col= "label",
                                    batch_size = batch_size,
                                    target_size=size, validate_filenames=False)

val_idg = idg.flow_from_dataframe(data_val, "datasets/val/", x_col= "filename", y_col= "label",
                                    batch_size = batch_size,
                                    target_size=size, validate_filenames=False)

Found 20000 non-validated image filenames belonging to 2 classes.
Found 5000 non-validated image filenames belonging to 2 classes.


In [7]:
vgg16_model = tf.keras.applications.vgg16.VGG16(include_top=False, input_shape=(370, 370, 3))

for layer in vgg16_model.layers:
    layer.trainable = False

In [8]:
flat = tf.keras.layers.Flatten() (vgg16_model.output)
dropout1 = tf.keras.layers.Dropout(0.2, name="Dropout1") (flat)
dense1 = tf.keras.layers.Dense(128, activation="relu") (dropout1)
dropout2 = tf.keras.layers.Dropout(0.2, name="Dropout2")(dense1)
output = tf.keras.layers.Dense(2, activation="softmax") (dropout2)

final_model = tf.keras.models.Model(inputs=[vgg16_model.input], outputs=[output])

In [9]:
final_model.compile(optimizer='adam',loss=tf.keras.losses.categorical_crossentropy,metrics = ["acc"])

In [10]:
learning_rate_reduction = ReduceLROnPlateau(monitor = 'val_accuracy', patience=2,
                                            factor=0.5, min_lr = 0.00001,
                                            verbose = 1)

early_stoping = EarlyStopping(monitor='val_loss', patience=5,
                              restore_best_weights=True, verbose=0)

In [12]:
history = final_model.fit(train_idg, 
                          batch_size=batch_size, 
                          validation_data=val_idg, 
                          epochs = 20, 
                          callbacks=[learning_rate_reduction,early_stoping])

Epoch 1/20
 73/313 ━━━━━━━━━━━━━━━━━━━━ 1:41:43 25s/step - acc: 0.8825 - loss: 3.3892

KeyboardInterrupt: 

In [ ]:
error = pd.DataFrame(history.history)
error.to_latex()

In [ ]:
error = pd.DataFrame(history.history)

plt.figure(figsize=(18,5),dpi=200)
sns.set_style('darkgrid')

plt.subplot(121)
plt.title('Cross Entropy Loss',fontsize=15)
plt.xlabel('Epochs',fontsize=12)
plt.ylabel('Loss',fontsize=12)
plt.plot(error['loss'])
plt.plot(error['val_loss'])

plt.subplot(122)
plt.title('Classification Accuracy',fontsize=15)
plt.xlabel('Epochs',fontsize=12)
plt.ylabel('Accuracy',fontsize=12)
plt.plot(error['acc'])
plt.plot(error['val_acc'])

plt.show()

In [ ]:
loss3,acc3 = final_model.evaluate(train_idg,batch_size = batch_size, verbose = 0)

print('The accuracy of the model for training data is:',acc3*100)
print('The Loss of the model for training data is:',loss3)

In [ ]:
loss3,acc3 = final_model.evaluate(val_idg,batch_size = batch_size, verbose = 0)

print('The accuracy of the model for validation data is:',acc3*100)
print('The Loss of the model for validation data is:',loss3)

In [ ]:
result = final_model.predict(test_idg)

result_argmax = np.argmax(result, axis=1)

y_true = test_idg.labels

y_pred = result_argmax

# Evaluvate
loss3,acc3 = final_model.evaluate(test_idg,batch_size = batch_size, verbose = 0)

print('The accuracy of the model for testing data is:',acc3*100)
print('The Loss of the model for testing data is:',loss3)

In [ ]:
y_true = test_idg.labels
print(repr(y_pred))
print(y_true)

In [ ]:
labels =['Cat','Dog']
print(classification_report(y_true, y_pred,target_names=labels))

classification_report(y_true, y_pred,target_names=labels)

In [ ]:
confusion_mtx = confusion_matrix(y_true,y_pred) 

f,ax = plt.subplots(figsize = (10,4),dpi=200)
sns.heatmap(confusion_mtx, annot=True, linewidths=0.1, cmap = "gist_yarg_r", linecolor="black", fmt='.0f', ax=ax,cbar=False,xticklabels=labels,yticklabels=labels)
plt.xlabel("Predicted Label",fontsize=10)
plt.ylabel("True Label",fontsize=10)
plt.title("Confusion Matrix",fontsize=13)
plt.show()

### CIFAR-10 Classification using Pretrained Model

In [ ]:
from keras.datasets import cifar10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()
assert x_train.shape == (50000, 32, 32, 3)
assert x_test.shape == (10000, 32, 32, 3)
assert y_train.shape == (50000, 1)
assert y_test.shape == (10000, 1)

In [ ]:
x_test, x_val,y_test, y_val = train_test_split(x_test, y_test, test_size=0.5, random_state = 23)

In [ ]:
from tensorflow.keras.utils import to_categorical

# Convert labels to one-hot encoding
y_train = to_categorical(y_train, num_classes=10)
y_val = to_categorical(y_val, num_classes=10)


In [ ]:
vgg16_model_cifar = tf.keras.applications.vgg16.VGG16(include_top=False, input_shape=(32, 32, 3))

for layer in vgg16_model_cifar.layers:
    layer.trainable = False

In [ ]:
flat = tf.keras.layers.Flatten() (vgg16_model_cifar.output)
dropout1 = tf.keras.layers.Dropout(0.2, name="Dropout1") (flat)
dense1 = tf.keras.layers.Dense(128, activation="relu") (dropout1)
dropout2 = tf.keras.layers.Dropout(0.2, name="Dropout2")(dense1)
output = tf.keras.layers.Dense(10, activation="softmax") (dropout2)

final_model_cifar = tf.keras.models.Model(inputs=[vgg16_model_cifar.input], outputs=[output])

In [ ]:
final_model_cifar.compile(optimizer='adam',loss=tf.keras.losses.categorical_crossentropy,metrics = ["acc"])

In [ ]:
learning_rate_reduction = ReduceLROnPlateau(monitor = 'val_acc', patience=2,
                                            factor=0.5, min_lr = 0.00001,
                                            verbose = 1)

early_stoping = EarlyStopping(monitor='val_loss', patience=5,
                              restore_best_weights=True, verbose=1)

In [ ]:
history_cifar = final_model_cifar.fit(x_train, y_train, 
                          batch_size=batch_size, 
                          validation_data=(x_val,y_val), 
                          epochs = 100, 
                          callbacks=[learning_rate_reduction,early_stoping])

In [ ]:
error_cifar = pd.DataFrame(history_cifar.history)

plt.figure(figsize=(18,5),dpi=200)
sns.set_style('darkgrid')

plt.subplot(121)
plt.title('Cross Entropy Loss',fontsize=15)
plt.xlabel('Epochs',fontsize=12)
plt.ylabel('Loss',fontsize=12)
plt.plot(error_cifar['loss'])
plt.plot(error_cifar['val_loss'])
plt.legend(['Training','Validation'])

plt.subplot(122)
plt.title('Classification Accuracy',fontsize=15)
plt.xlabel('Epochs',fontsize=12)
plt.ylabel('Accuracy',fontsize=12)
plt.plot(error_cifar['acc'])
plt.plot(error_cifar['val_acc'])
plt.legend(['Training','Validation'])
plt.savefig('çifar_accuracy.jpg')

plt.show()

In [ ]:
result_cifar = final_model_cifar.predict(x_test)

result_cifar_argmax = np.argmax(result_cifar, axis=1)

y_true = y_test

y_pred = result_cifar_argmax

In [ ]:
labels =['Airplane','Automobile','bird','çat', 'deer', 'dog','frog', 'horse', 'ship', 'truck']
print(classification_report(y_true, y_pred,target_names=labels))

classification_report(y_true, y_pred,target_names=labels)